# Perceptron and Neural Network Modeling

This notebook implements a simple Perceptron, a class-based version, the continuous sigmoid function, and an example of a multilayer neural network (MLP). The goal is to illustrate the progression from a basic linear model to a multi-layer architecture with nonlinear transformations.


## Basic Perceptron functions

In this block we define the core functions of a Perceptron:

- `sigma(v)`: step activation function, returning `1` if `v >= 0` and `0` otherwise.
- `predict(x, w, b)`: computes the Perceptron's prediction for input `x` by evaluating `w @ x + b` and applying the activation.
- `train(X, y, w, b, eta, epochs)`: implements the Perceptron learning algorithm. For each epoch and each pair `(x, y_true)`, it computes `y_hat`, measures the error `y_true - y_hat`, and updates weights `w` and bias `b` using the learning rate `eta`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def sigma(v):
    return 1 if v >= 0 else 0


def predict(x, w, b):
    return sigma(w @ x + b)


def train(X, y, w, b, eta, epochs=1000):
    for _ in range(epochs):
        for x, y_true in zip(X, y):
            y_hat = predict(x, w, b)
            w = w + eta * (y_true - y_hat) * x
            b = b + eta * (y_true - y_hat)
    return w, b


## Data initialization and initial prediction

Here we define the OR gate dataset, initialize weights and bias with arbitrary values, and compute the Perceptron's predictions before any training. We then use a pandas `DataFrame` to view inputs `(X1, X2)`, true labels `y`, and predictions `y_hat` side by side.


In [ ]:
# Input data (OR gate)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([0, 1, 1, 1])

# Weight and bias initialization
w = np.array([0.5, -1.5])
b = 2.0

# Initial prediction (no training)
y_hat = np.zeros(len(X))
for i in range(len(X)):
    y_hat[i] = predict(X[i, :], w, b)

pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Perceptron training

In this section, the Perceptron is trained on `X` and `y` with learning rate `eta = 0.1`. After training, we print the updated weights `w` and bias `b`, and for each sample we show `(y_true, y_hat)` to check whether the model has learned the OR function. Finally, we recompute `y_hat` and display a new `DataFrame` comparing true labels and predictions.


In [ ]:
# Reset weights and bias
w = np.array([0.5, -1.5])
b = 2.0

# Training loop
w, b = train(X, y, w, b, eta=0.1)
print("Trained parameters:", w, b)

# Evaluation after training
y_hat = np.zeros(len(X))
for i in range(len(X)):
    y_hat[i] = predict(X[i, :], w, b)
    print(f"x: {X[i]}  y_true: {y[i]}  y_hat: {y_hat[i]}")

pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Improved version: `Perceptron` class

To make the code more modular and reusable, we encapsulate the Perceptron in a class. The `Perceptron` class initializes `w` and `b`, stores the learning rate `eta` and number of epochs, and exposes methods to predict (`predict`) and train (`train`) the model using the same update rule as before.


In [ ]:
class Perceptron:
    def __init__(self, input_size, eta=0.1, epochs=1000):
        self.w = np.zeros(input_size)
        self.b = 0.0
        self.eta = eta
        self.epochs = epochs

    def getParam(self):
        return {'w': self.w, 'b': self.b}

    def sigma(self, v):
        return 1 if v >= 0 else 0

    def predict(self, x):
        return self.sigma(self.w @ x + self.b)

    def train(self, X, y):
        w, b, eta = self.w, self.b, self.eta
        epochs = self.epochs

        for _ in range(epochs):
            for x, y_true in zip(X, y):
                y_hat = self.predict(x)
                w = w + eta * (y_true - y_hat) * x
                b = b + eta * (y_true - y_hat)

        self.w, self.b = w, b


In [ ]:
p = Perceptron(input_size=2, eta=0.1, epochs=1000)

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([0, 1, 1, 1])

p.train(X, y)
print("Parameters:", p.getParam())

y_hat = np.array([p.predict(x) for x in X])
pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Continuous sigmoid function and derivative

Here we define the continuous sigmoid function, `sigma(z) = 1 / (1 + exp(-z))`, and its derivative `diff_sigma(z) = s * (1 - s)`, where `s = sigma(z)`. We evaluate them at a specific point and generate values over a grid in `[-5, 5]` to plot both the sigmoid curve and its derivative.


In [ ]:
def sigma(z):
    return 1.0 / (1.0 + np.exp(-z))


def diff_sigma(z):
    s = sigma(z)
    return s * (1 - s)

print("sigma(2.6) =", sigma(2.6))
print("diff_sigma(2.6) =", diff_sigma(2.6))

xx = np.linspace(-5, 5, 100)
zz = [sigma(x) for x in xx]
zz_diff = [diff_sigma(x) for x in xx]

plt.plot(xx, zz, label="sigmoid")
plt.plot(xx, zz_diff, label="derivative")
plt.legend()
plt.grid(True)
plt.show()


## Multilayer neural network (MLP) example

We now extend the Perceptron idea to a neural network with multiple layers. We define the input, hidden, and output layer sizes, initialize weight matrices `W1` and `W2` with random values in `[-1, 1]`, and perform a **forward pass**: compute hidden-layer activations and then output-layer activations using the sigmoid function. The continuous outputs are rounded to obtain binary predictions, and we display a `DataFrame` comparing `y` and `y_hat`.


In [ ]:
from numpy.random import rand

np.random.seed(42)


def sigma(z):
    return 1.0 / (1.0 + np.exp(-z))


def diff_sigma(z):
    s = sigma(z)
    return s * (1 - s)

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([0, 1, 1, 1])

input_size = 2
hidden_size = 3
output_size = 1

# Transform [0,1] to [-1,1]
W1 = 2 * rand(input_size, hidden_size) - 1
W2 = 2 * rand(hidden_size, output_size) - 1

# Batch forward pass
y_hat = np.zeros(len(X))

for i in range(len(X)):
    z1 = X[i] @ W1
    a1 = sigma(z1)

    z2 = a1 @ W2
    a2 = sigma(z2)

    y_hat[i] = np.round(a2)

pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Linear mapping between intervals with `encontra_coeficientes`

The `encontra_coeficientes` function finds coefficients `a` and `b` of a linear transformation `f(x) = a x + b` that maps an original interval `[x_min, x_max]` to a new interval `[y_min, y_max]`. We show examples of mapping `[0, 1]` to `[-1, 1]` and `[-10, 10]`, and apply the transformation `2 * X - 1` directly to the data.


In [ ]:
def encontra_coeficientes(intervalo_original, intervalo_mapeado):
    A = np.array([[intervalo_original[0], 1],
                  [intervalo_original[1], 1]])
    b = np.array([[intervalo_mapeado[0]],
                  [intervalo_mapeado[1]]])
    return np.linalg.solve(A, b)

print("Mapping [0,1] -> [-1,1]:", encontra_coeficientes([0, 1], [-1, 1]))
print("Mapping [0,1] -> [-10,10]:", encontra_coeficientes([0, 1], [-10, 10]))

print("2*X - 1:
", 2 * X - 1)
